In [1]:
!pip install -U transformers accelerate bitsandbytes sentencepiece tavily-python trafilatura langgraph langchain-core openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.15.0
    Uninstalling openai-2.15.0:
      Successfully uninstalled openai-2.15.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting unins

In [2]:
import warnings
warnings.filterwarnings('ignore')

import logging
import urllib3
logging.getLogger("trafilatura").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.ERROR)

import os
import json
import time
import random
import re

import pandas as pd
from typing import List, TypedDict, Literal

from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from tqdm.asyncio import tqdm as tqdm_async
import asyncio
from tavily import TavilyClient
import trafilatura

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    BaseMessage
)
from langgraph.graph import StateGraph, END

from openai import OpenAI, AsyncOpenAI, RateLimitError

from google.colab import userdata

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
MAX_CONCURRENT_REQUESTS = 1

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
data_path = "/content/drive/MyDrive/data_final_for_dls_new.jsonl"
data_df = pd.read_json(data_path, lines=True)

train_df = data_df[570:]
eval_df = data_df[:570]
eval_df = eval_df[eval_df["relevance"] != 0.1]

In [9]:
eval_df

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized,relevance_new
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите...",1.0
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан...",0.0
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос...",1.0
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...,1.0
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо...",1.0
...,...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...,1.0
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...,0.0
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None,0.0
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...,0.0


In [10]:
def prep_df(df):
    initial_len = len(df)

    df = df.drop_duplicates(subset=["Text", "name", "address"])

    ambiguous_count = len(df[~df["relevance"].isin([0.0, 1.0])])
    df = df[df["relevance"].isin([0.0, 1.0])]

    if ambiguous_count > 0:
        print(f"Удалено {ambiguous_count} строк с нечеткой релевантностью")

    if "relevance" in df.columns:
        df = df.drop(columns=["relevance"])

    if "relevance_new" in df.columns:
        df = df.rename(columns={"relevance_new": "relevance"})

    return df


train_df = prep_df(train_df)

Удалено 4633 строк с нечеткой релевантностью


In [56]:
try:
    TAVILY_KEY = userdata.get("Tavily")
except:
    TAVILY_KEY = input("Введите Tavily API Key: ")

try:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except:
    OPENROUTER_API_KEY = input("Введите OpenRouter API Key: ")

In [57]:
tavily_client = TavilyClient(api_key=TAVILY_KEY)

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

Baseline

In [ ]:
def build_baseline_prompt(row):
    return f"""
Ты — AI-классификатор релевантности.

ЗАДАЧА:
Определи, соответствует ли организация пользовательскому запросу.

ПРИМЕР 1:

ЗАПРОС: "недорогие кафе геленджика"
ОРГАНИЗАЦИЯ: "Весёлая Кума"
КАТЕГОРИЯ: "Кафе"

ОПИСАНИЕ:
"Уютное кафе на берегу моря с демократичными ценами."

ОТВЕТ:
{{
  "reasoning": "Организация является кафе, в описании указаны демократичные цены, что соответствует запросу.",
  "verdict": 1
}}

ПРИМЕР 2:

ЗАПРОС: "частная массажистка бутово"
ОРГАНИЗАЦИЯ: "Медицинский центр Здоровье"
КАТЕГОРИЯ: "Медцентр"

ОПИСАНИЕ:
"Медицинские услуги, массаж, физиотерапия."

ОТВЕТ:
{{
  "reasoning": "Запрос требует частную массажистку, а не медицинский центр.",
  "verdict": 0
}}

ТЕКУЩАЯ ЗАДАЧА:

ЗАПРОС: "{row['Text']}"
ОРГАНИЗАЦИЯ: "{row['name']}"
АДРЕС: "{row['address']}"
КАТЕГОРИЯ: "{row['normalized_main_rubric_name_ru']}"

ИНСТРУКЦИЯ:
- Если организация подходит запросу → verdict = 1
- Если не подходит → verdict = 0
- Отвечай СТРОГО в JSON
- НЕ используй markdown
- НЕ добавляй пояснений вне JSON

ФОРМАТ:
{{
  "reasoning": "...",
  "verdict": 0 или 1
}}
"""


In [ ]:
def baseline_llm_call(prompt: str, max_retries=5) -> int:
    for attempt in range(max_retries):
        try:
            response = openrouter_client.chat.completions.create(
                model="xiaomi/mimo-v2-flash:free",
                messages=[
                    {"role": "system", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=256,
            )

            content = response.choices[0].message.content
            content = content.replace("```json", "").replace("```", "").strip()

            data = json.loads(content)
            return int(data.get("verdict", 0))

        except RateLimitError:
            wait_time = 10 + attempt * 5
            print(f"Rate limit. Sleep {wait_time}s (retry {attempt+1}/{max_retries})")
            time.sleep(wait_time)

        except Exception as e:
            print("LLM error:", e)
            return 0

    print("Max retries exceeded, returning 0")
    return 0



In [ ]:
y_true = []
y_pred = []

print("\nBASELINE EVALUATION\n" + "-" * 60)

def clip(text: str, max_len: int = 30) -> str:
    text = str(text)
    return text if len(text) <= max_len else text[:max_len - 3] + "..."

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    prompt = build_baseline_prompt(row)
    pred = baseline_llm_call(prompt)
    gt = int(row["relevance"])

    y_true.append(gt)
    y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    query = row['Text']
    name = row['name']
    print(
        f"[{i:03d}] "
        f"Q: {clip(query, 35)} | "
        f"ORG: {clip(name, 25)} | "
        f"GT:{gt} P:{pred} | "
        f"ACC:{acc:.3f} F1:{f1:.3f}"
    )

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("-" * 60)
print(f"Baseline Accuracy: {acc:.3f}")
print(f"Baseline F1-score: {f1:.3f}")
print("-" * 60)



BASELINE EVALUATION
------------------------------------------------------------


  0%|          | 1/500 [00:09<1:21:35,  9.81s/it]

[000] Q: сигары | ORG: Tabaccos; Магазин Taba... | GT:1 P:1 | ACC:1.000 F1:1.000


  0%|          | 2/500 [00:24<1:46:46, 12.86s/it]

[001] Q: кальянная спб мероприятия | ORG: PioNero; Pionero; Пицц... | GT:0 P:0 | ACC:1.000 F1:1.000


...

[566] Q: домашний интернет в курске что п... | ORG: Цифровой канал; Digita... | GT:0 P:1 | ACC:0.627 F1:0.646


100%|█████████▉| 499/500 [2:13:04<00:13, 13.48s/it]

[567] Q: гостиница волгодонск сауна номер... | ORG: Поплавок; Poplavok | GT:0 P:1 | ACC:0.625 F1:0.645


100%|██████████| 500/500 [2:13:23<00:00, 16.01s/it]

[569] Q: футбол для подростков в нижнем н... | ORG: АСК | GT:0 P:1 | ACC:0.624 F1:0.644
------------------------------------------------------------
Baseline Accuracy: 0.624
Baseline F1-score: 0.644
------------------------------------------------------------


Agent

In [59]:
class AgentState(TypedDict):
    query: str
    org_name: str
    org_address: str
    org_category: str
    web_context: str
    iterations: int
    decision: dict

LLM_RPS = 0.25  # 1 запрос в 4 секунды (15 / мин)
llm_lock = asyncio.Lock()
last_call_time = 0.0

async def openrouter_llm_invoke_async(client, messages):
    global last_call_time

    async with llm_lock:
        now = time.time()
        wait = max(0, (1 / LLM_RPS) - (now - last_call_time))
        if wait > 0:
            await asyncio.sleep(wait)
        last_call_time = time.time()

        loop = asyncio.get_running_loop()

        openai_messages = [{
            "role": "user",
            "content": "\n".join(m.content for m in messages)
        }]

        def _call():
            return client.chat.completions.create(
                model="xiaomi/mimo-v2-flash:free",
                messages=openai_messages,
                temperature=0.1,
                max_tokens=256,
            )

        try:
            response = await loop.run_in_executor(None, _call)
            return AIMessage(content=response.choices[0].message.content)
        except Exception as e:
            logging.error(f"Agent LLM error: {e}")
            raise


def clean_query_logic(name, address):
    clean_name = name.split(";")[0].strip()
    parts = address.split(",")
    city = parts[0].strip()
    if "Россия" in city and len(parts) > 1:
        city = parts[1].strip()
    return f"{clean_name} {city} официальный сайт услуги"


def _sync_web_search(tavily_client, query: str) -> str:
    try:
        results = tavily_client.search(query, search_depth="advanced", max_results=2)
        urls = [r["url"] for r in results.get("results", [])]
        combined_text = ""

        for url in urls:
            if any(bad in url for bad in ["youtube", "vk.com"]): continue
            try:
                # Trafilatura - блокирующая операция, поэтому вызываем внутри executor'а
                downloaded = trafilatura.fetch_url(url)
                if downloaded:
                    text = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
                    if text:
                        combined_text += f"\n--- SOURCE: {url} ---\n{text[:600]}...\n"
            except: pass

        return combined_text if combined_text else "Информация не найдена."
    except Exception as e:
        return f"Ошибка поиска: {e}"

# Асинхронная обертка для поиска
async def tool_web_search_async(tavily_client, query: str):
    loop = asyncio.get_running_loop()
    # Запускаем блокирующий поиск в отдельном потоке
    return await loop.run_in_executor(None, _sync_web_search, tavily_client, query)


def build_agent_prompt(state: AgentState) -> str:
    return f"""
Ты — AI-классификатор релевантности организаций пользовательскому запросу.

ПРИМЕР 1:

ЗАПРОС: "недорогие кафе геленджика"
ОРГАНИЗАЦИЯ: "Весёлая Кума"
КАТЕГОРИЯ: "Кафе"
АДРЕС: "Геленджик"

ОТВЕТ:
{{
  "reasoning": "Кафе в Геленджике, соответствует запросу.",
  "verdict": 1,
  "confidence": 0.85
}}

ПРИМЕР 2:

ЗАПРОС: "частная массажистка бутово"
ОРГАНИЗАЦИЯ: "Медицинский центр Здоровье"
КАТЕГОРИЯ: "Медцентр"
АДРЕС: "Москва"

ОТВЕТ:
{{
  "reasoning": "Запрос требует частного специалиста, а не медцентр.",
  "verdict": 0,
  "confidence": 0.9
}}

ТЕКУЩАЯ ЗАДАЧА:

ЗАПРОС: "{state['query']}"
ОРГАНИЗАЦИЯ: "{state['org_name']}"
КАТЕГОРИЯ: "{state['org_category']}"
АДРЕС: "{state['org_address']}"

ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ (если есть):
{state['web_context'] if state['web_context'] else "Нет."}

ИНСТРУКЦИЯ:
- Определи, соответствует ли организация запросу
- Верни verdict = 1 или 0
- confidence ∈ [0, 1]
- Отвечай СТРОГО в JSON
- НЕ используй markdown
- НЕ добавляй текста вне JSON

ФОРМАТ:
{{
  "reasoning": "...",
  "verdict": 0 или 1,
  "confidence": 0.0–1.0
}}
"""

def create_async_agent_graph(tavily_client, llm_client: AsyncOpenAI):

    async def node_analyst(state: AgentState):
        prompt = build_agent_prompt(state)

        response = await openrouter_llm_invoke_async(
            llm_client,
            [SystemMessage(content=prompt)]
        )

        try:
            content = response.content.strip().replace("```", "")
            decision = json.loads(content)
        except Exception:
            decision = {
                "verdict": 0,
                "confidence": 0.0,
                "reasoning": "JSON parse error"
            }

        action = "finish"
        if state["iterations"] == 0 and decision["confidence"] < 0.6:
            action = "search"
        else:
            action = "finish"

        decision["action"] = action

        if state["iterations"] >= 1:
            decision["action"] = "finish"

        return {
            "decision": decision,
            "iterations": state["iterations"] + 1
        }

    async def node_searcher(state: AgentState):
        query = clean_query_logic(
            state["org_name"],
            state["org_address"]
        )

        web_info = await tool_web_search_async(tavily_client, query)

        return {
            "web_context": f"=== SEARCH RESULT ===\n{web_info[:1200]}"
        }

    def router(state: AgentState) -> Literal["search_tool", "end"]:
        if state["decision"].get("action") == "search":
            return "search_tool"
        return "end"

    workflow = StateGraph(AgentState)

    workflow.add_node("analyst", node_analyst)
    workflow.add_node("search_tool", node_searcher)

    workflow.set_entry_point("analyst")

    workflow.add_conditional_edges(
        "analyst",
        router,
        {
            "search_tool": "search_tool",
            "end": END
        }
    )

    workflow.add_edge("search_tool", "analyst")

    return workflow.compile()

In [60]:
async def process_agent_row(agent_app, row, semaphore):
    """Обработка одной строки агентом с семафором"""
    async with semaphore:
        inputs = {
            "query": row["Text"],
            "org_name": row["name"],
            "org_address": row["address"],
            "org_category": row["normalized_main_rubric_name_ru"],
            "web_context": "",
            "iterations": 0,
            "thoughts": [],
            "decision": {}
        }
        # ainvoke - асинхронный вызов графа
        res = await agent_app.ainvoke(inputs)
        pred = int(res["decision"].get("verdict", 0))
        return int(row["relevance"]), pred

In [64]:
semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)


async def main():

    agent_app = create_async_agent_graph(tavily_client, openrouter_client)

    agent_tasks = []
    for i, row in eval_df[:25].iterrows():
        agent_tasks.append(process_agent_row(agent_app, row, semaphore))

    # Запускаем все задачи агента параллельно
    results = await asyncio.gather(*agent_tasks)

    y_true_agent = [r[0] for r in results]
    y_pred_agent = [r[1] for r in results]

    acc = accuracy_score(y_true_agent, y_pred_agent)
    f1 = f1_score(y_true_agent, y_pred_agent, zero_division=0)

    return acc, f1

In [72]:
acc, f1 = await main()

In [73]:
print(f"Agent Accuracy: {acc:.3f}")
print(f"Agent F1: {f1:.3f}")

Agent Accuracy: 0.625
Agent F1: 0.664
